<a href="https://colab.research.google.com/github/siva123995/ProjectLLM/blob/main/ReviewAgentLinkedInPost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Inspired by vamsiBhavani.in

Exercise 2: build an agent that should accept the topic, and write content on it to post in a LinkedIn, also another agent should review it, and rephrase the content based on the review, stop the process after N times.

give the last post to post in LinkedIn.

In [1]:
# installing the required libraries
!pip install -qU langgraph langchain_google_genai langchain_community langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
langchain 1.2.15 requi

In [24]:
# getting the secrets
from google.colab import userdata
GEMINI_API_KEY=userdata.get('Secret')
# GEMINI_API_KEY=userdata.get('sivathede')

In [25]:
# import model.
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model = "models/gemini-3-flash-preview", google_api_key = GEMINI_API_KEY)
# response = model.invoke('Hello World')
# print(response.content)

In [8]:
import operator
from typing import Annotated, List, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

In [26]:
class AgentState(TypedDict):
  topic : str
  review: str
  revision_count : Annotated[int, operator.add]

writer_prompt = ChatPromptTemplate.from_template(
    "write a post based on topic: {topic}"
)

writer_chain = writer_prompt | model

reviewer_prompt = ChatPromptTemplate.from_template(
    "Review the post and suggest me improvements to post in LinkedIn: {post}"
)

reviewer_chain  = reviewer_prompt | model

def writer_node(state: AgentState):
  topic = state['topic']

  prompt_content = {"topic": topic}
  if state.get('review'):
    prompt_content['topic'] += f"Also consider these suggestions {state['review']}"

  post_content = writer_chain.invoke(prompt_content).content

  return {"post":post_content, "revision_count":1 }

def reviewer_node(state: AgentState):
  post = state.get('post')

  prompt_content = {"post": post}
  review_content = reviewer_chain.invoke(prompt_content).content
  return {"review": review_content}

graph = StateGraph(AgentState)
graph.add_node("writer", writer_node)
graph.add_node("reviewer", reviewer_node)

def conditional_check(state: AgentState):
  if state['revision_count'] >=3:
    return END
  return "reviewer"

graph.add_edge(START, "writer")
graph.add_edge("writer","reviewer")
graph.add_conditional_edges("reviewer", conditional_check)
app=graph.compile()

In [28]:
response = app.invoke({"topic":"Job market in Queensland Australia"})
for k,v in response.items():
  print(f"key--> {k} value-> {v}")

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 48.234682727s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPer

## Conversation Summary and Knowledge Discussed

This conversation has focused on setting up, debugging, and understanding a `langgraph` application designed to create and review LinkedIn posts. The goal is to have an AI writer generate a post, an AI reviewer suggest improvements, and the writer revise the post based on feedback, repeating this process a specified number of times.

### Key Information and Concepts:

1.  **Objective:** Build a `langgraph` agent that accepts a `topic`, writes a LinkedIn `post`, gets it `review`ed, `revises` it, and `stops` after a certain number of `revisions` (N times).

2.  **Environment Setup:**
    *   Required libraries (`langgraph`, `langchain_google_genai`, `langchain_community`, `langchain_core`) were installed.
    *   `GEMINI_API_KEY` was loaded from Colab userdata.
    *   A `ChatGoogleGenerativeAI` model (`gemini-3-flash-preview`) was initialized.

3.  **`langgraph` State Management (`AgentState`):**
    *   The `AgentState(TypedDict)` is the central shared memory/context for the graph, defining its schema.
    *   It specifies the variables that will be passed between nodes, such as `topic`, `post`, `review`, and `revision_count`.
    *   **`state['key']` vs. `state.get('key')`:**
        *   `state['key']` is used when the key is *guaranteed* to exist. If it doesn't, a `KeyError` is raised.
        *   `state.get('key')` is used when the key *might not* exist. It returns `None` (or a specified default) if the key is absent, preventing errors.

4.  **Reducer Functions with `Annotated`:**
    *   `Annotated[Type, operator]` allows `langgraph` to apply custom merge logic when updating state variables.
    *   **`Annotated[int, operator.add]` for `revision_count`:**
        *   This ensures that instead of overwriting, new `revision_count` values are *added* to the existing one.
        *   If `revision_count` is not explicitly initialized in the `invoke` call, `langgraph` automatically initializes it to `0` (the identity element for `operator.add`).
    *   **`Annotated[List[str], operator.add]` for `review`:**
        *   This was discussed as a way to accumulate reviews as a list, rather than overwriting them. If a node returns `{"review": ["new review"]}`, it appends to the existing list.

5.  **Graph Structure:**
    *   `writer_prompt` and `reviewer_prompt` (using `ChatPromptTemplate`) are defined for the LLM interactions.
    *   `writer_chain` and `reviewer_chain` are created by piping prompts to the `model`.
    *   **`writer_node`:** Takes the `state` (including `topic` and potentially `review`), generates a `post`, and returns `{"post": post_content, "revision_count": 1}`.
    *   **`reviewer_node`:** Takes the `post` from the `state`, generates a `review`, and returns `{"review": review_content}`.
    *   **`conditional_check`:** Routes the graph.
        *   If `state['revision_count'] >= 3`, it returns `END` (terminates).
        *   Otherwise, it returns `"reviewer"`, looping back for another revision.
    *   Edges defined: `START -> Writer`, `Writer -> reviewer`, and `reviewer` conditionally routes back to `Writer` or `END`.

6.  **Debugging and Issues Resolved:**
    *   **`ModuleNotFoundError` for `langchain_core.templates`:** Corrected to `langchain_core.prompts`.
    *   **`ImportError` for `StateGraph`:** Corrected to `from langgraph.graph import StateGraph`.
    *   **`TypeError: StateGraph.__init__() missing 1 required positional argument: 'state_schema'`:** Resolved by passing `AgentState` to the `StateGraph` constructor.
    *   **`InvalidUpdateError: Expected dict, got [HumanMessage(...)]`:** Input to `app.invoke()` corrected to be a dictionary matching `AgentState`.
    *   **`AttributeError: 'AIMessage' object has no attribute 'invoke'`:** Corrected `reviewer_chain.invoke(prompt_content).invoke` to `reviewer_chain.invoke(prompt_content).content` to access the AI's message content.
    *   **`TypeError: '>=' not supported between instances of 'NoneType' and 'int'`:** This was due to a typo (`revison_count` vs. `revision_count`) in `AgentState` and the return value of `writer_node`, which caused `revision_count` to remain `None`. Correcting the key name ensured `Annotated` could properly initialize and increment the count.

7.  **Performance/Quota:** Noted that the graph runs longer due to multiple LLM calls per revision and the configured `revision_count` threshold. A `RESOURCE_EXHAUSTED` error indicated hitting the API rate limits, requiring a retry after a cooldown period.

In [ ]:
import operator
from typing import Annotated, TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END


writer_prompt = ChatPromptTemplate.from_template(
    "write a LinkedIn post on topic: {topic}"
)
writer_chain = writer_prompt | model

reviewer_prompt = ChatPromptTemplate.from_template(
    "Review following LinkedIn post and suggest me improvements:{post}"
)
reviewer_chain = reviewer_prompt | model

class AgentState(TypedDict):
    topic: str
    post: str
    review: str # Added review to state
    revision_count: Annotated[int, operator.add] # Use Annotated for automatic increment

graph = StateGraph(AgentState)

def writerNode(state: AgentState) -> dict:
  topic = state['topic']
  # If there's a review, incorporate it into the prompt for revision
  prompt_input = {"topic": topic}
  if state.get('review'):
      prompt_input["topic"] += f" (incorporate these suggestions: {state['review']})"

  post_content = writer_chain.invoke(prompt_input).content
  # This node updates 'post' and increments 'revision_count' by 1
  return {"post": post_content, "revision_count": 1}

def reviewer_node(state: AgentState) -> dict:
  post = state['post']
  review_content = reviewer_chain.invoke({"post": post}).content
  return {"review": review_content} # Update 'review' in state

graph.add_node("Writer", writerNode)
graph.add_node("reviewer", reviewer_node)

graph.add_edge(START, "Writer")
graph.add_edge("Writer", "reviewer") # After writing, go to reviewer

def condition_check(state: AgentState):
  # Terminate if revision_count exceeds a threshold (e.g., 3 revisions total)
  if state['revision_count'] >= 3: # Allow max 3 revisions
    return END
  # Otherwise, go back to Writer for revision
  return "Writer"

graph.add_conditional_edges("reviewer", condition_check) # After reviewer, decide where to go


app = graph.compile()

if you would like to acumulate reviews as well

In [19]:
import operator
from typing import Annotated, TypedDict, List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END


writer_prompt = ChatPromptTemplate.from_template(
    "write a LinkedIn post on topic: {topic}"
)
writer_chain = writer_prompt | model

reviewer_prompt = ChatPromptTemplate.from_template(
    "Review following LinkedIn post and suggest me improvements:{post}"
)
reviewer_chain = reviewer_prompt | model

class AgentState(TypedDict):
    topic: str
    post: str
    review: Annotated[List[str], operator.add] # Changed to accumulate reviews as a list
    revision_count: Annotated[int, operator.add] # Use Annotated for automatic increment

graph = StateGraph(AgentState)

def writerNode(state: AgentState) -> dict:
  topic = state['topic']
  # If there's a review, incorporate it into the prompt for revision
  prompt_input = {"topic": topic}
  if state.get('review') and len(state['review']) > 0: # Check if review list exists and is not empty
      # Join all accumulated reviews for the prompt
      all_reviews = " ".join(state['review'])
      prompt_input["topic"] += f" (incorporate these suggestions: {all_reviews})"

  post_content = writer_chain.invoke(prompt_input).content
  # This node updates 'post' and increments 'revision_count' by 1
  return {"post": post_content, "revision_count": 1}

def reviewer_node(state: AgentState) -> dict:
  post = state['post']
  review_content = reviewer_chain.invoke({"post": post}).content
  return {"review": [review_content]} # Return the new review as a list to append

graph.add_node("Writer", writerNode)
graph.add_node("reviewer", reviewer_node)

graph.add_edge(START, "Writer")
graph.add_edge("Writer", "reviewer") # After writing, go to reviewer

def condition_check(state: AgentState):
  # Terminate if revision_count exceeds a threshold (e.g., 3 revisions total)
  if state['revision_count'] >= 3: # Allow max 3 revisions
    return END
  # Otherwise, go back to Writer for revision
  return "Writer"

graph.add_conditional_edges("reviewer", condition_check) # After reviewer, decide where to go


app = graph.compile()

In [20]:
response = app.invoke({"topic": "Job Market in India"}) # Correct input: dict matching AgentState
print("Final State:")
for k, v in response.items():
  print(f"{k}: {v}") # Print items from the final state dictionary

Final State:
topic: Job Market in India
post: [{'type': 'text', 'text': 'Here is a high-authority LinkedIn post tailored for the Indian job market, incorporating the strategic shifts, visionary tone, and formatting tips you provided.\n\n***\n\n**Headline: The 2021 job market wasn\'t "normal." It was an anomaly. 🛑**\n\nI’m seeing a lot of frustration from talented professionals in India right now. The experience is there, the degrees are there, but the offers aren\'t hitting the inbox like they used to.\n\n**The reality?** The Indian job market isn’t broken—it’s evolving. 📈\n\nWe have officially shifted from **Growth-at-all-costs** (Volume) to **Efficiency-at-all-costs** (Value). India is no longer just the "back office" of the world; we are becoming the "global cockpit."\n\nIf you’re navigating this market, stop looking in the rearview mirror at 2021 and look where the puck is going:\n\n✅ **The GCC Revolution:** \n1,500+ Global Capability Centers are no longer here just for support. Th

In [22]:
print("Starting the graph execution with streaming...")
accumulated_state = {"topic": "Job Market in India", "post": "", "review": [], "revision_count": 0} # Initialize review as an empty list

for s in app.stream({"topic": "Job Market in India"}, {'recursion_limit': 10}): # Added recursion_limit for safety
    for key, value in s.items():
        # Update the accumulated_state with the changes from the current step
        # Langgraph handles Annotated[operator.add] automatically for simple types
        if key == '__end__': # Handle the final state from the compiler
            accumulated_state.update(value)
        else:
            # Langgraph's operator.add for lists will automatically append
            # For other non-Annotated types, it will just overwrite or update the dict.
            # This update logic needs to be careful for how each field is defined.
            # For a TypedDict with Annotated values, `accumulated_state.update(value)` will internally handle the reducer for `Annotated` fields.
            if key in accumulated_state and isinstance(accumulated_state[key], list) and isinstance(value, list):
                # This explicit append is needed because update() for lists with operator.add is implicit in StateGraph processing
                # when not dealing with the direct StateGraph's state updates in the loop. A simpler way is to let StateGraph manage it.
                # However, for manual accumulation in the stream loop for display, we must handle lists carefully if not using the actual State object.
                # Let's rely on `accumulated_state = graph.get_state_history()` if we need perfect intermediate state representations.
                # For now, let's update simply and expect `operator.add` to be managed by `langgraph` internally.
                # A safer approach for displaying the *current* state of the graph during streaming (not just the delta) is to get the full state after each step
                pass # The StateGraph handles accumulation, we just need to get the latest full state.
            else:
                accumulated_state.update(value)

    # After each step, the accumulated_state should reflect the current state correctly
    # However, `s` provides delta. To get the actual state, one might need `graph.get_state_history()`
    # For this display, we'll continue using the previous logic for updates to `accumulated_state`.

    if "Writer" in s:
        print("\n--- Writer Node Output ---")
        print(f"Post: {accumulated_state['post']}") # Print from accumulated state
        print(f"Current Revision Count: {accumulated_state['revision_count']}") # Print from accumulated state
    if "reviewer" in s:
        print("\n--- Reviewer Node Output ---")
        print(f"Review: {accumulated_state['review']}") # Print from accumulated state
        print(f"Current Revision Count: {accumulated_state['revision_count']}") # Print from accumulated state

# The last yielded item 's' will contain the final state when the loop ends
final_state = accumulated_state

print("\n--- Final State ---")
for k, v in final_state.items():
  print(f"{k}: {v}")

Starting the graph execution with streaming...

--- Writer Node Output ---
Post: [{'type': 'text', 'text': 'To help you get the best engagement, I’ve drafted three different types of posts depending on the "vibe" you want to project: **The Optimist (Growth-focused)**, **The Realistic Advisor (Career advice)**, and **The Trend Watcher (Data-driven).**\n\n---\n\n### Option 1: The Optimist (Focus on GCCs and India as a Global Hub)\n**Best for:** Establishing yourself as a forward-thinking professional.\n\n**Headline:** India is no longer just the world’s "back office." 🇮🇳\n\nWe are witnessing a massive shift in the Indian job market. The rise of Global Capability Centers (GCCs) has changed the game. \n\nInternational giants aren’t just looking for cost-saving anymore; they are looking for **high-value leadership, R&D, and innovation.**\n\nWhat does this mean for us?\n🚀 **More Specialized Roles:** Demand for AI, Cloud, and Cybersecurity experts is at an all-time high.\n🚀 **Leadership Oppor